In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [(1.1732+1.3325)/2, 1.1732+1.3325] # First value is the average of the two photopeaks. They have the same branching ratio


colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kViolet, ROOT.kYellow, ROOT.kSpring, ROOT.kCyan,]

In [2]:
files = sorted((project_root/'data/One_sided_new_data').glob('SUBT_*'))
acqs = [SiphraAcquisition(file) for file in files]
print(acqs)

[SIPHRAAcquisition(File: 'SUBT_15_LongBackgroundAtEnd'), SIPHRAAcquisition(File: 'SUBT_16_Cs137OnCrystalAtCenter'), SIPHRAAcquisition(File: 'SUBT_18_Co60OnCrystalAtCenter'), SIPHRAAcquisition(File: 'SUBT_20_Na22OnCrystalAtCenter')]


In [3]:
# histograms
hists = {}
sources = []
siphra = 'A'


bg = acqs[0]

for sgnl in acqs[1:]:
    filepath = sgnl.filepath.stem
    src = (MetadataLoader.load(sgnl.metadataFile)).source
    print(src)
    sources.append(src)
    # Signal and Background
    hist_sgnl = ROOT.TH1F(f"{src} Signal, SIPHRA {siphra}", "", BITS12, 0, BITS12)
    hist_bg = ROOT.TH1F(f"{src} Background, SIPHRA {siphra}", "", BITS12, 0 , BITS12)
    if siphra == 'Summed':
        hist_sgnl.Fill(sgnl['Combined_Summed']/(len(sgnl.active_chs)*2))
        hist_bg.Fill(bg['Combined_Summed']/(len(bg.active_chs)*2))
    else:
        hist_sgnl.Fill(sgnl['s']/len(sgnl.active_chs))
        hist_bg.Fill(bg['s']/len(bg.active_chs))
    hist_bg.Scale(sgnl.exposure/bg.exposure)
    # Clean spectrum
    hist_clean = hist_sgnl.Clone(f"{src} BG subtracted, SIPHRA {siphra}")
    hist_clean.Add(hist_bg, -1)
    for hist in [hist_sgnl, hist_bg, hist_clean]:
        # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
        hist.GetXaxis().SetTitle("ADC channel number")
        hist.GetYaxis().SetTitle(r"Normalized counts")
    hists[f"{src}_SIPHRA_{siphra}"] = hist_sgnl
    hists[f"{src}_BG_SIPHRA_{siphra}"] = hist_bg
    hists[f"{src}_clean_SIPHRA_{siphra}"] = hist_clean
    del hist_sgnl, hist_bg

Cs-137
Co-60
Na-22


In [4]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 3200, 2400)
cv.Divide(2, 2)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(4)]


for idx, src in enumerate(sources):   
    cv.cd(idx+1)
    lg = lgds[idx]
    cv.cd(idx+1).SetLogy()

    print(src)
    sgnl = hists[f"{src}_SIPHRA_{siphra}"]
    bg = hists[f"{src}_BG_SIPHRA_{siphra}"] 
    clean = hists[f"{src}_clean_SIPHRA_{siphra}"]
    lg.AddEntry(sgnl, "Signal")
    lg.AddEntry(bg, "Background")
    lg.AddEntry(clean, "Subtracted")
    sgnl.SetLineColor(colors[0])
    bg.SetLineColor(colors[1])
    clean.SetLineColor(colors[2])
    sgnl.SetTitle(f"One-sided readout, {src}")
    sgnl.Draw("hist")
    bg.Draw("sames hist")
    clean.Draw("sames hist")
    ROOT.gPad.Update()
    #for j, sp in enumerate([sgnl, bg, clean]):
    #    st = sp.FindObject("stats")
        # print(type(st))
    #    st.SetY1NDC(Yinit - j*0.08)
    #    st.SetY2NDC(0.1)
    lg.Draw()
    
    cv.Draw()

Cs-137
Co-60
Na-22


In [5]:
from analysis import *
resolution_cs = energy_resolution(hists[f"Cs-137_SIPHRA_A"], [(380, 560)])
resolution_co = energy_resolution(hists[f"Co-60_SIPHRA_A"], [(1530, 1780)])
resolution_na = energy_resolution(hists[f"Na-22_SIPHRA_A"], [(280, 440), (760, 930), (1200, 1280)])




Resolution: 20.2 %
Resolution Error: 0.0354 %
Resolution: 10.7 %
Resolution Error: 0.313 %
Resolution: 31.7 %
Resolution Error: 0.291 %
Resolution: 21.9 %
Resolution Error: 0.902 %
Resolution: 15.8 %
Resolution Error: 5.19 %
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      372.641
NDf                       =          177
Edm                       =  3.44999e-08
NCalls                    =           73
Constant                  =       3396.7   +/-   7.69914     
Mean                      =      459.608   +/-   0.0765029   
Sigma                     =      39.5077   +/-   0.0676234    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      297.346
NDf                       =          246
Edm                       =  1.20519e-05
NCalls                    =           79
Constant                  =       15.208   +/-   0.399801    
Mean                      =      1632.16   +/

In [6]:
from analysis import *

#Calibration based on the 3 peaks of the Na-22 spectrum
src = "Na-22"

hist = hists[f"{src}_clean_SIPHRA_{siphra}"]
energy_ranges = [(280, 440), (760, 930), (1200, 1280)]
energies = NA22_MEV

c_fit = calibration_fit(hist, energy_ranges, energies)
print(c_fit)

[0.0016095714089286292, -0.07826372966375345]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      159.744
NDf                       =          157
Edm                       =  1.21825e-06
NCalls                    =           71
Constant                  =      227.542   +/-   2.18717     
Mean                      =      364.601   +/-   0.404859    
Sigma                     =      44.0254   +/-   0.44011      	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      160.719
NDf                       =          167
Edm                       =  1.68288e-10
NCalls                    =           92
Constant                  =      34.6464   +/-   0.81382     
Mean                      =      844.257   +/-   1.79208     
Sigma                     =      66.5102   +/-   2.69815      	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2             

In [7]:
#Calibrating Cs137 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[0], BITS12)
hist_cal_bg_Cs137.Scale(1/acqs[0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[1], BITS12)
hist_cal_src_Cs137.Scale(1/acqs[1].exposure)
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)


ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)
#cv1.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
#cv1.cd(1)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])


lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{137}Cs signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{137}Cs background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
hist_cal_src_Cs137.SetTitle(r"One-sided readout ^{137}Cs")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()


#res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(0.45, 0.81)])
#print(res_Cs137)



Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [10]:
#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Na22 = calibrated_histogram(c_fit, acqs[0], BITS12)
hist_cal_bg_Na22.Scale(1/acqs[0].exposure) 
hist_cal_src_Na22 = calibrated_histogram(c_fit, acqs[3], BITS12)
hist_cal_src_Na22.Scale(1/acqs[3].exposure)
hist_cal_clean_Na22 = hist_cal_src_Na22.Clone("Calibrated signal no BG")
hist_cal_clean_Na22.Add(hist_cal_bg_Na22, -1)


ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv2'):
    ROOT.gROOT.FindObject('cv2').Close()
cv2 = ROOT.TCanvas("cv2", "cv2", 1200, 800)

lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Na22.SetLineColor(colors[1])
hist_cal_src_Na22.SetLineColor(colors[0])
hist_cal_clean_Na22.SetLineColor(colors[2])


lg2.AddEntry(hist_cal_bg_Na22, "Background", "l")
lg2.AddEntry(hist_cal_src_Na22, r"^{22}Na signal", "l")
lg2.AddEntry(hist_cal_clean_Na22, r"^{22}Na background subtracted", "l")
hist_cal_src_Na22.Draw("hist")
hist_cal_bg_Na22.Draw("sames hist")
hist_cal_clean_Na22.Draw("sames hist")
hist_cal_src_Na22.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Na22.GetYaxis().SetTitle(r"Count rate [Hz]")
hist_cal_src_Na22.SetTitle(r"One-sided readout ^{22}Na")
lg2.Draw()
cv2.SetLogy()
cv2.Draw()


#res_Na22 = energy_resolution(hist_cal_clean_Na22, [(0.28, 0.73), (1.13, 1.45), (1.58, 1.95)])
#print(res_Na22)


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [11]:
#Calibrating Co60 histograms based on Na22 calibration fit
hist_cal_bg_Co60 = calibrated_histogram(c_fit, acqs[0], BITS12)
hist_cal_bg_Co60.Scale(1/acqs[0].exposure) 
hist_cal_src_Co60 = calibrated_histogram(c_fit, acqs[2], BITS12)
hist_cal_src_Co60.Scale(1/acqs[2].exposure)
hist_cal_clean_Co60 = hist_cal_src_Co60.Clone("Calibrated signal no BG")
hist_cal_clean_Co60.Add(hist_cal_bg_Co60, -1)


ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv3'):
    ROOT.gROOT.FindObject('cv3').Close()
cv3 = ROOT.TCanvas("cv3", "cv3", 1200, 800)

lg3 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Co60.SetLineColor(colors[1])
hist_cal_src_Co60.SetLineColor(colors[0])
hist_cal_clean_Co60.SetLineColor(colors[2])


lg3.AddEntry(hist_cal_bg_Co60, "Background", "l")
lg3.AddEntry(hist_cal_src_Co60, r"^{60}Co signal", "l")
lg3.AddEntry(hist_cal_clean_Co60, r"^{60}Co background subtracted", "l")
hist_cal_src_Co60.Draw("hist")
hist_cal_bg_Co60.Draw("sames hist")
hist_cal_clean_Co60.Draw("sames hist")
hist_cal_src_Co60.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Co60.GetYaxis().SetTitle(r"Count rate [Hz]")
hist_cal_src_Co60.SetTitle(r"One-sided readout ^{60}Co")
lg3.Draw()
cv3.SetLogy()
cv3.Draw()


#res_Co60 = energy_resolution(hist_cal_clean_Co60, [(1.0, 1.4), (2.3, 2.65)] )
#print(res_Co60)

Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
